In [32]:
!pip install PyEMD river scikit-learn torch numpy pandas matplotlib seaborn -q

In [2]:
# CELL 1 — Install Dependencies
!pip install EMD-signal -q        # PyEMD is published as EMD-signal on PyPI
!pip install river -q
!pip install scikit-learn -q
!pip install torch numpy pandas matplotlib seaborn -q

# Verify import works
from PyEMD import CEEMDAN
print("PyEMD import OK ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires dill<0.3.9,>=0.3.0, but you have dill 0.4.1 which is incompatible.
datasets 4.0.0 requires multiprocess<0.70.17, but you have multiprocess 0.70.19 which is incompatible.
PyEMD import OK ✅


In [70]:
import os, gc, math, copy, time, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from PyEMD import CEEMDAN
from river.drift import ADWIN
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")

# ── Reproducibility seed (will be overridden per run) ──────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[CONFIG] Device : {DEVICE}")

# ── Rolling / Window Parameters ────────────────────────────────────────
WINDOW_SIZE      = 300        # Rolling window (trading days)
ROLLING_ERR_WIN  = 100        # Error monitoring window for drift
VOL_WINDOW       = 30         # Days for log-return volatility
VOL_THRESHOLD    = 0.006
VOL_K = 50.0        # controls steepness of sigmoid transition
W_MIN = 0.20       # minimum Informer weight
W_MAX = 0.80       # maximum Informer weight     # Calibrated to actual data: range 0.004-0.007, mean 0.005
                               # 0.006 = ~75th percentile → realistic HIGH/LOW split
SEQ_LEN          = 60         # Input sequence length (fallback)
MAX_IMF          = 8          # CEEMDAN max decomposition levels
N_RERUNS         = 5          # Robustness reruns (req: 5)
LOG_EVERY        = 10         # Print every 10 steps so you see live progress

# ── CEEMDAN Smart Caching ──────────────────────────────────────────────
# Adjacent 300-day windows differ by only 1 day, so IMF structure
# changes minimally between consecutive steps.
# Recomputing every 5 steps cuts CEEMDAN cost by ~5x with negligible
# accuracy loss. Drift events always force an immediate recompute.
CEEMDAN_RECOMPUTE_EVERY = 5

# ── Adaptive Weighting Bounds ──────────────────────────────────────────
# Change these in Cell 2:
W_H_HIGH_VOL = 0.52   # was 0.70 — slight Informer boost in high vol
W_H_LOW_VOL  = 0.48   # was 0.35 — slight Informer reduction in low vol

# ── Drift / Retraining ─────────────────────────────────────────────────
ADWIN_DELTA      = 0.001
DRIFT_STD_MULT   = 2.0        # Secondary guard: error > μ + k·σ
MAX_RETRAINS     = 2
FINETUNE_EPOCHS  = 2
FINETUNE_LR      = 1e-5
RETRAIN_EPOCHS   = 18
RETRAIN_LR       = 1e-4
BATCH_SIZE       = 16

# ── Phase 1 Model Specs (from requirements PDF) ────────────────────────
# Informer: d_model=256, enc_layers=2, dec_layers=1, batch=32, lr=1e-4
# LSTM    : look_back=20, depth=4, hidden=4 units (per paper), epochs=100
# (These are used only for fallback random-init if .pt missing)

# ── IMF → Model Mapping ────────────────────────────────────────────────
IMF_MODEL_MAP = {
    "IMF1":     ("informer", "informer_IMF1.pt"),
    "IMF2":     ("informer", "informer_IMF2.pt"),
    "IMF3":     ("informer", "informer_IMF3.pt"),
    "IMF4":     ("lstm",     "lstm_IMF4.pt"),
    "IMF5":     ("lstm",     "lstm_IMF5.pt"),
    "IMF6":     ("informer", "informer_IMF6.pt"),
    "IMF7":     ("informer", "informer_IMF7.pt"),
    "IMF8":     ("lstm",     "lstm_IMF8.pt"),
    "Residual": ("lstm",     "lstm_Residual.pt"),
}

INFORMER_COMPS = [k for k, (t, _) in IMF_MODEL_MAP.items() if t == "informer"]
LSTM_COMPS     = [k for k, (t, _) in IMF_MODEL_MAP.items() if t == "lstm"]

[CONFIG] Device : cuda


In [71]:
class _FixedPE(nn.Module):
    """Positional encoding stored as buffer (key: self.pe)."""
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class _MHA(nn.Module):
    """Multi-head attention with Phase 1 key names: q_proj, k_proj, v_proj, out_proj."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.h      = n_heads
        self.dh     = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.drop   = nn.Dropout(dropout)

    def _split(self, x):
        B, L, _ = x.shape
        return x.view(B, L, self.h, self.dh).transpose(1, 2)

    def forward(self, q, k, v):
        Q = self._split(self.q_proj(q))
        K = self._split(self.k_proj(k))
        V = self._split(self.v_proj(v))
        sc = (Q @ K.transpose(-2, -1)) * (self.dh ** -0.5)
        w  = self.drop(sc.softmax(-1))
        o  = (w @ V).transpose(1, 2).contiguous().view(q.shape[0], q.shape[1], -1)
        return self.out_proj(o)


class _EncoderLayer(nn.Module):
    """
    Matches checkpoint keys:
      encoder.N.attn.{q,k,v,out}_proj
      encoder.N.ff.0  (Linear d_model→d_ff)
      encoder.N.ff.3  (Linear d_ff→d_model)   ← index 3 not 2
      encoder.N.norm1, encoder.N.norm2
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn  = _MHA(d_model, n_heads, dropout)
        # ff.0 and ff.3 to match Phase 1 (ReLU is index 1, Dropout index 2)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_ff),   # ff.0
            nn.ReLU(),                   # ff.1
            nn.Dropout(dropout),         # ff.2
            nn.Linear(d_ff, d_model),   # ff.3
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        x = self.norm1(x + self.drop(self.attn(x, x, x)))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x


class _DecoderLayer(nn.Module):
    """
    Matches checkpoint keys:
      decoder.0.self_attn.{q,k,v,out}_proj
      decoder.0.cross_attn.{q,k,v,out}_proj
      decoder.0.ff.0 / ff.3
      decoder.0.norm1, norm2, norm3
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = _MHA(d_model, n_heads, dropout)
        self.cross_attn = _MHA(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, tgt, mem):
        tgt = self.norm1(tgt + self.drop(self.self_attn(tgt, tgt, tgt)))
        tgt = self.norm2(tgt + self.drop(self.cross_attn(tgt, mem, mem)))
        tgt = self.norm3(tgt + self.drop(self.ff(tgt)))
        return tgt


class InformerIMF(nn.Module):
    """
    Full Encoder-Decoder Informer matching Phase 1 checkpoint EXACTLY.
    Key mapping:
      enc_embed (Linear 1→d_model)
      dec_embed (Linear 1→d_model)
      enc_pos.pe  (1, enc_seq_len, d_model)
      dec_pos.pe  (1, dec_seq_len, d_model)
      encoder.N.*   (_EncoderLayer ×n_enc_layers)
      decoder.0.*   (_DecoderLayer ×1)
      enc_norm, dec_norm  (LayerNorm)
      proj  (Linear d_model→1)
    """
    def __init__(self,
                 enc_seq_len  = 96,
                 dec_seq_len  = 48,
                 d_model      = 256,
                 n_heads      = 16,
                 d_ff         = 1024,
                 n_enc_layers = 2,
                 n_dec_layers = 1,
                 dropout      = 0.1):
        super().__init__()
        self.seq_len     = enc_seq_len
        self.dec_seq_len = dec_seq_len
        self.pred_len    = 1

        self.enc_embed = nn.Linear(1, d_model)
        self.dec_embed = nn.Linear(1, d_model)
        self.enc_pos   = _FixedPE(d_model, enc_seq_len + 10, dropout)
        self.dec_pos   = _FixedPE(d_model, dec_seq_len + 10, dropout)

        self.encoder   = nn.ModuleList(
            [_EncoderLayer(d_model, n_heads, d_ff, dropout)
             for _ in range(n_enc_layers)]
        )
        self.decoder   = nn.ModuleList(
            [_DecoderLayer(d_model, n_heads, d_ff, dropout)
             for _ in range(n_dec_layers)]
        )
        self.enc_norm  = nn.LayerNorm(d_model)
        self.dec_norm  = nn.LayerNorm(d_model)
        self.proj      = nn.Linear(d_model, 1)   # proj: (1, 256)

    def forward(self, x):
        # x: (B, enc_seq_len, 1)
        B = x.size(0)

        # Encoder
        mem = self.enc_pos(self.enc_embed(x))
        for layer in self.encoder:
            mem = layer(mem)
        mem = self.enc_norm(mem)

        # Decoder — use LAST dec_seq_len steps of INPUT as start token
        # This is the standard Informer inference pattern (not zeros)
        dec_input = x[:, -self.dec_seq_len:, :]          # (B, dec_L, 1)
        dec_token = self.dec_pos(self.dec_embed(dec_input))
        for layer in self.decoder:
            dec_token = layer(dec_token, mem)
        dec_token = self.dec_norm(dec_token)

        # Last decoder step → scalar prediction
        out = self.proj(dec_token[:, -1, :])              # (B, 1)
        return out


class LSTMIMF(nn.Module):
    """LSTM used for Low-Frequency IMFs and Residual."""
    def __init__(self, seq_len=20, pred_len=1, hidden=64,
                 num_layers=4, dropout=0.1):
        super().__init__()
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.lstm = nn.LSTM(1, hidden, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc   = nn.Linear(hidden, pred_len)

    def forward(self, x):
        o, _ = self.lstm(x)
        return self.fc(o[:, -1, :])


In [72]:
def _make_informer_from_ckpt(state: dict) -> InformerIMF:
    """
    Build InformerIMF whose dimensions are read directly from checkpoint.
    enc_seq_len  ← enc_pos.pe shape[1] - 10
    dec_seq_len  ← dec_pos.pe shape[1] - 10
    d_model      ← enc_embed.weight shape[0]
    d_ff         ← encoder.0.ff.0.weight shape[0]
    n_heads      ← largest divisor of d_model in {16,8,4,2,1}
    n_enc_layers ← count of encoder.N keys
    n_dec_layers ← count of decoder.N keys
    """
    d_model      = state["enc_embed.weight"].shape[0]
    d_ff         = state["encoder.0.ff.0.weight"].shape[0]
    enc_seq_len  = state["enc_pos.pe"].shape[1] - 10
    dec_seq_len  = state["dec_pos.pe"].shape[1] - 10

    enc_indices  = set(int(k.split(".")[1]) for k in state if k.startswith("encoder."))
    dec_indices  = set(int(k.split(".")[1]) for k in state if k.startswith("decoder."))
    n_enc_layers = max(enc_indices) + 1 if enc_indices else 2
    n_dec_layers = max(dec_indices) + 1 if dec_indices else 1

    n_heads = 16
    for h in [16, 8, 4, 2, 1]:
        if d_model % h == 0:
            n_heads = h
            break

    return InformerIMF(
        enc_seq_len  = max(enc_seq_len, 10),
        dec_seq_len  = max(dec_seq_len, 10),
        d_model      = d_model,
        n_heads      = n_heads,
        d_ff         = d_ff,
        n_enc_layers = n_enc_layers,
        n_dec_layers = n_dec_layers,
        dropout      = 0.1,
    )


def _make_lstm_from_ckpt(state: dict) -> LSTMIMF:
    hidden     = state["lstm.weight_hh_l0"].shape[1]
    num_layers = len([k for k in state
                      if "weight_hh_l" in k and "_reverse" not in k])
    pred_len   = state["fc.weight"].shape[0]
    return LSTMIMF(seq_len=20, pred_len=pred_len,
                   hidden=hidden, num_layers=num_layers, dropout=0.1)


def load_phase1_models(model_dir: str = "/content/") -> dict:
    """
    Load all 9 Phase 1 pretrained models with STRICT weight loading.
    Architecture is built to exactly match each checkpoint.
    """
    models = {}
    print("\n[MODELS] Loading Phase 1 pretrained weights...")
    for comp, (mtype, fname) in IMF_MODEL_MAP.items():
        path = os.path.join(model_dir, fname)
        print(f"  [LOAD] {comp:10s} <- {fname}")
        if not os.path.exists(path):
            print(f"       [WARN] Not found -> random init")
            model = InformerIMF() if mtype == "informer" else LSTMIMF()
            models[comp] = model.to(DEVICE).eval()
            continue

        raw   = torch.load(path, map_location=DEVICE)
        state = raw.get("model_state_dict", raw) if isinstance(raw, dict) else raw

        try:
            if mtype == "informer":
                model = _make_informer_from_ckpt(state)
                print(f"       enc_seq={model.seq_len} dec_seq={model.dec_seq_len} "
                      f"d_model={model.enc_embed.weight.shape[0]} "
                      f"n_enc={len(model.encoder)} n_dec={len(model.decoder)}")
            else:
                model = _make_lstm_from_ckpt(state)
                print(f"       hidden={model.lstm.hidden_size} "
                      f"layers={model.lstm.num_layers}")

            model.load_state_dict(state, strict=True)
            print(f"       STRICT load OK")

        except Exception as e:
            print(f"       [ERR] {e}")
            print(f"       -> Using random init as fallback")
            model = InformerIMF() if mtype == "informer" else LSTMIMF()

        models[comp] = model.to(DEVICE).eval()

    print(f"[MODELS] All {len(models)} models ready on {DEVICE}\n")
    return models


In [73]:
def run_ceemdan(signal: np.ndarray, max_imf: int = MAX_IMF) -> dict:
    ceemdan = CEEMDAN(trials=20, epsilon=0.005)
    imfs    = ceemdan(signal)          # ← call directly, returns array
    result  = {}
    for i in range(max_imf):
        key = f"IMF{i + 1}"
        result[key] = imfs[i] if i < len(imfs) else np.zeros_like(signal)
    recon              = sum(result[f"IMF{i+1}"] for i in range(max_imf))
    result["Residual"] = signal - recon
    return result

def rolling_log_vol(close: np.ndarray, window: int = VOL_WINDOW) -> float:
    """
    Compute rolling volatility as std-dev of log-returns over last `window` days.
    Matches paper formula: σ_t = std(log-returns over 30 days)
    """
    if len(close) < window + 1:
        return 0.0
    recent = close[-(window + 1):]
    lr     = np.diff(np.log(recent + 1e-8))
    return float(np.std(lr))


def compute_hybrid_weights(vol: float, vol_mean: float, vol_std: float) -> tuple[float, float]:

    # Standardize volatility
    z = (vol - vol_mean) / (vol_std + 1e-8)

    # Sharper sigmoid
    sigmoid = 1.0 / (1.0 + np.exp(-5.0 * z))

    w_H = 0.20 + (0.80 - 0.20) * sigmoid
    w_L = 1.0 - w_H

    return float(w_H), float(w_L)


In [74]:
def make_sequences(imf_dict: dict, seq_len: int) -> dict:
    """
    Build supervised (X, y) pairs for each IMF component.
    Returns dict: comp → (X_tensor(N,L,1), y_tensor(N,1), scaler)
    """
    out = {}
    for comp, arr in imf_dict.items():
        scaler = MinMaxScaler(feature_range=(-1, 1))
        scaled = scaler.fit_transform(arr.reshape(-1, 1)).flatten()
        Xl, yl = [], []
        for i in range(len(scaled) - seq_len):
            Xl.append(scaled[i: i + seq_len])
            yl.append(scaled[i + seq_len])
        if not Xl:
            out[comp] = None
            continue
        X = torch.tensor(np.array(Xl), dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(np.array(yl), dtype=torch.float32).unsqueeze(-1)
        out[comp] = (X, y, scaler)
    return out


def get_seq_len(model: nn.Module) -> int:
    """Extract seq_len the model was built with."""
    if hasattr(model, "seq_len"):
        return max(model.seq_len, 10)
    if hasattr(model, "pos_enc") and hasattr(model.pos_enc, "pe"):
        return max(model.pos_enc.pe.shape[1] - 20, 10)
    return SEQ_LEN


def predict_component(model: nn.Module, imf_arr: np.ndarray, scaler) -> float:
    sl = get_seq_len(model)
    # Pad or trim to exactly seq_len
    if len(imf_arr) >= sl:
        last_seq = imf_arr[-sl:]
    else:
        last_seq = np.pad(imf_arr, (sl - len(imf_arr), 0), mode="edge")

    model.eval()
    with torch.no_grad():
        sc  = scaler.transform(last_seq.reshape(-1, 1)).flatten()
        x   = torch.tensor(sc, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(DEVICE)
        # x shape: (1, seq_len, 1) — correct for both Informer and LSTM
        out = model(x).cpu().numpy().flatten()[0]
    return float(scaler.inverse_transform([[out]])[0][0])


In [75]:
def finetune_model(model, X, y, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR):
    """Light per-step fine-tune on most recent batch (2 epochs, lr=1e-5)."""
    model.train()
    opt  = optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    Xb   = X[-BATCH_SIZE:].to(DEVICE)
    yb   = y[-BATCH_SIZE:].to(DEVICE)
    for _ in range(epochs):
        opt.zero_grad()
        crit(model(Xb), yb).backward()
        opt.step()
    return model


def full_retrain_model(model, X, y, epochs=RETRAIN_EPOCHS, lr=RETRAIN_LR):
    """Full retrain on entire current window (18 epochs, lr=1e-4)."""
    model.train()
    dl   = DataLoader(TensorDataset(X.to(DEVICE), y.to(DEVICE)),
                      batch_size=BATCH_SIZE, shuffle=True)
    opt  = optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    for _ in range(epochs):
        for xb, yb in dl:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
    return model

In [76]:
class DriftDetector:
    """
    ADWIN drift detection with statistical guard.
    Drift triggers ONLY when:
      ① ADWIN detects significant mean shift
      AND
      ② |error| > rolling_mean + DRIFT_STD_MULT * rolling_std
    Minor fluctuations are ignored.
    """
    def __init__(self, delta=ADWIN_DELTA):
        self.adwin   = ADWIN(delta=delta)
        self.history = []
        self.total   = 0

    def update(self, error: float) -> bool:
        ae = abs(error)
        self.history.append(ae)
        self.adwin.update(ae)
        adwin_flag = self.adwin.drift_detected
        if len(self.history) >= 10:
            arr = np.array(self.history[-ROLLING_ERR_WIN:])
            stat_flag = ae > arr.mean() + DRIFT_STD_MULT * arr.std()
        else:
            stat_flag = False
        if adwin_flag and stat_flag:
            self.total += 1
            return True
        return False

    def reset(self):
        self.adwin = ADWIN(delta=ADWIN_DELTA)

    @property
    def drift_count(self):
        return self.total

In [77]:


def static_predict(models_snapshot: dict, imf_dict: dict,
                   seq_dict: dict) -> float:
    """
    Static baseline: fixed equal weights (0.5/0.5) on group SUMS.
    Ŷ = l_sum + 0.5 * h_sum
    No adaptation, no drift response.
    """
    h_sum, l_sum = 0.0, 0.0
    for comp, model in models_snapshot.items():
        if seq_dict.get(comp) is None:
            continue
        _, _, sc = seq_dict[comp]
        pred = predict_component(model, imf_dict[comp], sc)
        if comp in INFORMER_COMPS:
            h_sum += pred
        else:
            l_sum += pred
    return l_sum + 0.5 * h_sum

In [91]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10 ── Main Rolling Adaptive Forecast (FINAL STABLE VERSION)
# ═══════════════════════════════════════════════════════════════════════

def run_single_adaptive_forecast(
    close: np.ndarray,
    dates: np.ndarray,
    models_init: dict,
    run_id: int = 1,
) -> dict:

    set_seed(42 + run_id * 7)

    models        = {k: copy.deepcopy(v).to(DEVICE).eval()
                     for k, v in models_init.items()}
    static_models = {k: copy.deepcopy(v).to(DEVICE).eval()
                     for k, v in models_init.items()}

    detector          = DriftDetector()
    retrain_count     = 0
    prev_imfs         = None
    prev_seqs         = None
    last_ceemdan_step = -CEEMDAN_RECOMPUTE_EVERY

    records   = []
    event_log = []
    n_steps   = len(close) - WINDOW_SIZE - 1

    print(f"\n{'='*65}")
    print(f"  RUN {run_id}/{N_RERUNS}  |  {n_steps} rolling steps  |  Device: {DEVICE}")
    print(f"{'='*65}")
    t0 = time.time()

    # ───────────── Precompute volatility distribution ──────────────
    vol_samples = []
    for i in range(n_steps):
        win_end = i + WINDOW_SIZE
        window  = close[i: win_end]
        vol_samples.append(rolling_log_vol(window))

    vol_samples = np.array(vol_samples)
    vol_mean    = float(np.mean(vol_samples))
    vol_std     = float(np.std(vol_samples))

    # ────────────────────────────────────────────────────────────────

    for step in range(n_steps):

        win_end   = step + WINDOW_SIZE
        window    = close[step: win_end]
        actual    = close[win_end]
        pred_date = dates[win_end]

        adaptive_pred       = float(window[-1])
        static_pred         = float(window[-1])
        retrained_this_step = False

        # ── Volatility & Adaptive Weight ────────────────────────────
        vol    = rolling_log_vol(window)
        regime = classify_regime(vol)

        w_H, w_L = compute_hybrid_weights(vol, vol_mean, vol_std)
        effective_w = 0.5 + 0.3 * (w_H - 0.5)

        # ── CEEMDAN Caching ─────────────────────────────────────────
        need_recompute = (step - last_ceemdan_step) >= CEEMDAN_RECOMPUTE_EVERY

        if need_recompute or prev_imfs is None:
            try:
                imf_dict = run_ceemdan(window)
                seq_dict = {}
                for comp, model in models.items():
                    sl    = get_seq_len(model)
                    built = make_sequences({comp: imf_dict[comp]}, seq_len=sl)
                    seq_dict[comp] = built.get(comp, None)
                prev_imfs         = imf_dict
                prev_seqs         = seq_dict
                last_ceemdan_step = step
            except:
                if prev_imfs is None:
                    continue
                imf_dict = prev_imfs
                seq_dict = prev_seqs
        else:
            imf_dict = prev_imfs
            seq_dict = prev_seqs

        # ── Component Predictions ───────────────────────────────────
        h_sum, l_sum = 0.0, 0.0

        for comp, model in models.items():
            if seq_dict.get(comp) is None:
                continue
            _, _, sc = seq_dict[comp]
            p = predict_component(model, imf_dict[comp], sc)

            if comp in INFORMER_COMPS:
                h_sum += p
            else:
                l_sum += p

        # ── Corrected Bias Scaling (CONSISTENT) ─────────────────────
        window_mean = float(np.mean(window[-20:]))

        raw_base = l_sum + effective_w * h_sum  # ← FIXED HERE

        scale = float(np.clip(window_mean / raw_base, 0.5, 2.0)) \
                if abs(raw_base) > 1e-6 else 1.0

        adaptive_pred = raw_base * scale  # ← FIXED HERE

        # Compute real error BEFORE drift
        error = actual - adaptive_pred

        # ── Static Baseline (unchanged) ─────────────────────────────
        try:
            sh, sl_ = 0.0, 0.0
            for comp, model in static_models.items():
                if seq_dict.get(comp) is None:
                    continue
                _, _, sc = seq_dict[comp]
                p = predict_component(model, imf_dict[comp], sc)
                if comp in INFORMER_COMPS:
                    sh += p
                else:
                    sl_ += p

            raw_static = sl_ + 0.5 * sh
            scale_s    = float(np.clip(window_mean / raw_static, 0.5, 2.0)) \
                         if abs(raw_static) > 1e-6 else 1.0
            static_pred = raw_static * scale_s
        except:
            static_pred = adaptive_pred

        # ── Drift Detection ─────────────────────────────────────────
        drift = detector.update(error)

        if drift and retrain_count < MAX_RETRAINS:
            retrain_count += 1
            retrained_this_step = True

            imf_dict = run_ceemdan(window)
            seq_dict = {}
            for comp, model in models.items():
                sl    = get_seq_len(model)
                built = make_sequences({comp: imf_dict[comp]}, seq_len=sl)
                seq_dict[comp] = built.get(comp, None)

            for comp, model in models.items():
                if seq_dict.get(comp) is not None:
                    X, y, _ = seq_dict[comp]
                    if len(X) > 0:
                        models[comp] = full_retrain_model(model, X, y)

            detector.reset()

            # Recompute prediction correctly after retrain
            h2, l2 = 0.0, 0.0
            for comp, model in models.items():
                if seq_dict.get(comp) is None:
                    continue
                _, _, sc = seq_dict[comp]
                p = predict_component(model, imf_dict[comp], sc)
                if comp in INFORMER_COMPS:
                    h2 += p
                else:
                    l2 += p

            raw_base2 = l2 + effective_w * h2   # ← FIXED
            scale2    = float(np.clip(window_mean / raw_base2, 0.5, 2.0)) \
                        if abs(raw_base2) > 1e-6 else 1.0
            adaptive_pred = raw_base2 * scale2
            error = actual - adaptive_pred

        # ── Record ───────────────────────────────────────────────────
        records.append(dict(
            Date        = pred_date,
            Actual      = actual,
            Predicted   = adaptive_pred,
            Error       = error,
            Static_Pred = static_pred,
            Volatility  = vol,
            Regime      = regime,
            W_H         = w_H,
            W_L         = w_L,
            Drift       = int(drift),
            Retrained   = int(retrained_this_step),
        ))

        if (step + 1) % LOG_EVERY == 0:
            pct = 100 * (step + 1) / n_steps
            print(f"[{pct:5.1f}%] Step {step+1}/{n_steps} | "
                  f"Regime={regime} w_H={w_H:.2f} | "
                  f"Drift#={detector.drift_count} Retrain#={retrain_count}")

    df = pd.DataFrame(records)
    df["Date"] = pd.to_datetime(df["Date"])

    return dict(
        forecast=df,
        events=pd.DataFrame(event_log),
        drift_count=detector.drift_count,
        retrain_count=retrain_count
    )

In [92]:
def directional_accuracy(actual: np.ndarray, predicted: np.ndarray) -> float:
    """Fraction of steps where predicted direction matches actual direction."""
    a_dir = np.diff(actual)
    p_dir = np.diff(predicted)
    return float(np.mean(np.sign(a_dir) == np.sign(p_dir)) * 100)


def compute_all_metrics(actual: np.ndarray,
                        predicted: np.ndarray,
                        label: str = "") -> dict:
    mask  = actual != 0
    mape  = float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)
    rmse  = float(np.sqrt(mean_squared_error(actual, predicted)))
    mae   = float(mean_absolute_error(actual, predicted))
    r2    = float(r2_score(actual, predicted))
    da    = directional_accuracy(actual, predicted)
    return dict(Model=label, RMSE=rmse, MAE=mae, MAPE=mape, R2=r2, DA=da)


def aggregate_runs(run_results: list, col: str = "Predicted") -> dict:
    """
    Compute Mean ± Std across N runs for all metrics.
    Returns dict of metric → (mean, std).
    """
    metrics_list = []
    for r in run_results:
        df = r["forecast"]
        m  = compute_all_metrics(df["Actual"].values,
                                 df[col].values, label=col)
        metrics_list.append(m)
    keys = ["RMSE", "MAE", "MAPE", "R2", "DA"]
    agg  = {}
    for k in keys:
        vals    = [m[k] for m in metrics_list]
        agg[k]  = (float(np.mean(vals)), float(np.std(vals)))
    return agg



In [93]:
def print_results_table(adaptive_agg: dict, static_agg: dict):
    print("\n" + "═" * 70)
    print("  PHASE 2  —  ADAPTIVE vs STATIC HYBRID  (Mean ± Std, 5 Runs)")
    print("═" * 70)
    header = f"  {'Metric':10s} {'Adaptive Hybrid':25s} {'Static Hybrid':25s}"
    print(header)
    print("─" * 70)
    for k in ["RMSE", "MAE", "MAPE", "R2", "DA"]:
        am, as_ = adaptive_agg[k]
        sm, ss  = static_agg[k]
        unit    = "%" if k in ("MAPE", "DA") else ""
        print(f"  {k:10s}  {am:.5f}{unit} ± {as_:.5f}   "
              f"{sm:.5f}{unit} ± {ss:.5f}")
    print("═" * 70)


def print_run_summary(run_results: list):
    print("\n" + "─" * 50)
    print(f"  {'Run':4s}  {'Drifts':7s}  {'Retrains':9s}")
    print("─" * 50)
    for i, r in enumerate(run_results, 1):
        print(f"  {i:4d}  {r['drift_count']:7d}  {r['retrain_count']:9d}")
    print("─" * 50)


def build_regime_table(run_results: list) -> pd.DataFrame:
    """Regime-wise metric breakdown (average across runs)."""
    rows = []
    for regime in ["HIGH", "LOW"]:
        r_metrics_a, r_metrics_s = [], []
        for r in run_results:
            df = r["forecast"]
            sub = df[df["Regime"] == regime]
            if len(sub) < 5:
                continue
            r_metrics_a.append(compute_all_metrics(
                sub["Actual"].values, sub["Predicted"].values))
            r_metrics_s.append(compute_all_metrics(
                sub["Actual"].values, sub["Static_Pred"].values))
        for col_m, label in [(r_metrics_a, "Adaptive"), (r_metrics_s, "Static")]:
            if not col_m:
                continue
            row = {"Regime": regime, "Model": label}
            for k in ["RMSE", "MAE", "MAPE", "R2", "DA"]:
                row[k] = float(np.mean([m[k] for m in col_m]))
            rows.append(row)
    return pd.DataFrame(rows)


In [94]:
def plot_phase2_results(run_results: list,
                        adaptive_agg: dict,
                        static_agg: dict,
                        save_dir: str = "/content/"):
    # Use the first run's forecast for time-series plots
    df       = run_results[0]["forecast"]
    ev       = run_results[0]["events"]
    dates    = df["Date"].values
    actual   = df["Actual"].values
    adaptive = df["Predicted"].values
    static   = df["Static_Pred"].values

    sns.set_theme(style="whitegrid", palette="muted")
    fig = plt.figure(figsize=(20, 22))
    gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.42, wspace=0.3)

    # ── Plot 1: Actual vs Adaptive vs Static ──────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(dates, actual,   label="Actual",          lw=1.4, color="#1f4e79")
    ax1.plot(dates, adaptive, label="Adaptive Hybrid", lw=1.0,
             color="#e74c3c", linestyle="--", alpha=0.88)
    ax1.plot(dates, static,   label="Static Hybrid",   lw=1.0,
             color="#27ae60", linestyle=":",  alpha=0.75)
    if len(ev) > 0 and "Date" in ev.columns:
        for _, row in ev.iterrows():
            ax1.axvline(pd.to_datetime(row["Date"]),
                        color="orange", alpha=0.6, linewidth=1.2, linestyle="-.")
    rmse_a = adaptive_agg["RMSE"][0]
    rmse_s = static_agg["RMSE"][0]
    ax1.set_title(
        f"Actual vs Predicted — Adaptive RMSE={rmse_a:.4f} | Static RMSE={rmse_s:.4f}",
        fontsize=12)
    ax1.legend(fontsize=9); ax1.set_xlabel("Date"); ax1.set_ylabel("Close Price")

    # ── Plot 2: Absolute Error Comparison ─────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.fill_between(dates, np.abs(df["Error"]), alpha=0.55,
                     color="#c0392b", label="|Adaptive Error|")
    ax2.fill_between(dates,
                     np.abs(actual - static), alpha=0.4,
                     color="#27ae60", label="|Static Error|")
    ax2.set_title("Absolute Prediction Error"); ax2.legend(fontsize=9)

    # ── Plot 3: Rolling Volatility & Regime ───────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(dates, df["Volatility"], color="#8e44ad", lw=1.2, label="30d Log-Return σ")
    ax3.axhline(VOL_THRESHOLD, color="red", linestyle="--", lw=0.9,
                label=f"Threshold={VOL_THRESHOLD}")
    high_mask = df["Regime"] == "HIGH"
    ax3.fill_between(dates, 0, df["Volatility"].max(),
                     where=high_mask, alpha=0.15, color="red", label="High-Vol Regime")
    ax3.set_title("Rolling Volatility & Regime Classification")
    ax3.legend(fontsize=8); ax3.set_ylabel("σ (log-return)")

    # ── Plot 4: Dynamic Hybrid Weights ────────────────────────────────
    ax4 = fig.add_subplot(gs[2, 0])
    ax4.plot(dates, df["W_H"], label="w_H (Informer)", color="#e67e22", lw=1.1)
    ax4.plot(dates, df["W_L"], label="w_L (LSTM)",     color="#2980b9", lw=1.1)
    ax4.set_ylim(0, 1); ax4.set_yticks([0, 0.35, 0.5, 0.65, 0.70, 1.0])
    ax4.set_title("Adaptive Hybrid Aggregation Weights  (w_H + w_L = 1)")
    ax4.legend(fontsize=9); ax4.set_ylabel("Weight")

    # ── Plot 5: Scatter Actual vs Predicted ───────────────────────────
    ax5 = fig.add_subplot(gs[2, 1])
    ax5.scatter(actual, adaptive, s=6, alpha=0.4, color="#e74c3c", label="Adaptive")
    ax5.scatter(actual, static,   s=6, alpha=0.3, color="#27ae60", label="Static")
    lo, hi = min(actual.min(), adaptive.min()), max(actual.max(), adaptive.max())
    ax5.plot([lo, hi], [lo, hi], "k--", lw=0.9, label="Perfect fit")
    ax5.set_title("Scatter: Actual vs Predicted"); ax5.legend(fontsize=8)
    ax5.set_xlabel("Actual"); ax5.set_ylabel("Predicted")

    # ── Plot 6: Metric Bar Chart (Adaptive vs Static) ─────────────────
    ax6 = fig.add_subplot(gs[3, :])
    metrics_labels = ["RMSE", "MAE", "MAPE (%)", "R²", "DA (%)"]
    a_vals = [adaptive_agg["RMSE"][0], adaptive_agg["MAE"][0],
              adaptive_agg["MAPE"][0], adaptive_agg["R2"][0], adaptive_agg["DA"][0]]
    s_vals = [static_agg["RMSE"][0],  static_agg["MAE"][0],
              static_agg["MAPE"][0],  static_agg["R2"][0],  static_agg["DA"][0]]
    x      = np.arange(len(metrics_labels))
    bw     = 0.35
    b1 = ax6.bar(x - bw/2, a_vals, bw, label="Adaptive Hybrid", color="#e74c3c", alpha=0.85)
    b2 = ax6.bar(x + bw/2, s_vals, bw, label="Static Hybrid",   color="#27ae60", alpha=0.75)
    ax6.bar_label(b1, fmt="%.4f", fontsize=7, padding=2)
    ax6.bar_label(b2, fmt="%.4f", fontsize=7, padding=2)
    ax6.set_xticks(x); ax6.set_xticklabels(metrics_labels)
    ax6.set_title("Adaptive vs Static Hybrid — Metric Comparison (Mean over 5 Runs)")
    ax6.legend(fontsize=9)

    plt.suptitle("Phase 2: CEEMDAN–Informer–LSTM Adaptive Rolling Forecast\n"
                 "NIFTY 50 Daily Close", fontsize=13, fontweight="bold", y=1.01)
    out = os.path.join(save_dir, "phase2_adaptive_results.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[PLOT] Saved → {out}")

In [95]:
def save_all_outputs(run_results: list,
                     adaptive_agg: dict,
                     static_agg: dict,
                     regime_df: pd.DataFrame,
                     save_dir: str = "/content/"):

    # 1. Full forecast log (last run, most detailed)
    forecast_path = os.path.join(save_dir, "phase2_forecast_log.csv")
    run_results[-1]["forecast"].to_csv(forecast_path, index=False)
    print(f"[SAVE] Forecast log     → {forecast_path}")

    # 2. Event log (drift / retrain events across all runs)
    all_events = []
    for i, r in enumerate(run_results, 1):
        if len(r["events"]) > 0:
            ev = r["events"].copy()
            ev.insert(0, "Run", i)
            all_events.append(ev)
    if all_events:
        ev_path = os.path.join(save_dir, "phase2_event_log.csv")
        pd.concat(all_events, ignore_index=True).to_csv(ev_path, index=False)
        print(f"[SAVE] Event log        → {ev_path}")

    # 3. Metrics summary
    rows = []
    for metric in ["RMSE", "MAE", "MAPE", "R2", "DA"]:
        am, as_ = adaptive_agg[metric]
        sm, ss  = static_agg[metric]
        rows.append(dict(Metric=metric,
                         Adaptive_Mean=am, Adaptive_Std=as_,
                         Static_Mean=sm,   Static_Std=ss))
    metrics_path = os.path.join(save_dir, "phase2_metrics_summary.csv")
    pd.DataFrame(rows).to_csv(metrics_path, index=False)
    print(f"[SAVE] Metrics summary  → {metrics_path}")

    # 4. Regime-wise breakdown
    if len(regime_df) > 0:
        reg_path = os.path.join(save_dir, "phase2_regime_analysis.csv")
        regime_df.to_csv(reg_path, index=False)
        print(f"[SAVE] Regime analysis  → {reg_path}")

    # 5. Per-run robustness table
    rob_rows = []
    for i, r in enumerate(run_results, 1):
        df = r["forecast"]
        am = compute_all_metrics(df["Actual"].values, df["Predicted"].values)
        sm = compute_all_metrics(df["Actual"].values, df["Static_Pred"].values)
        rob_rows.append(dict(Run=i,
                             **{f"Adaptive_{k}": am[k] for k in ["RMSE","MAE","MAPE","R2","DA"]},
                             **{f"Static_{k}":   sm[k] for k in ["RMSE","MAE","MAPE","R2","DA"]},
                             Drifts=r["drift_count"],
                             Retrains=r["retrain_count"]))
    rob_path = os.path.join(save_dir, "phase2_robustness_5runs.csv")
    pd.DataFrame(rob_rows).to_csv(rob_path, index=False)
    print(f"[SAVE] Robustness table → {rob_path}")


In [96]:
if __name__ == "__main__":

    CSV_PATH  = "/content/clean_daily_close.csv"
    MODEL_DIR = "/content/"
    SAVE_DIR  = "/content/"

    # ── 1. Load & validate full dataset ─────────────────────────────
    print("[DATA] Loading dataset…")
    df = pd.read_csv(CSV_PATH, parse_dates=["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df = df[["Date", "Close"]].dropna()
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    df = df.dropna().reset_index(drop=True)

    print(f"[DATA] Total rows   : {len(df)}")
    print(f"[DATA] Date range   : {df['Date'].iloc[0].date()} "
          f"→ {df['Date'].iloc[-1].date()}")
    print(f"[DATA] Close range  : {df['Close'].min():.2f} "
          f"→ {df['Close'].max():.2f}")

    # ── 2. Carve out Phase 2 evaluation slice ───────────────────────
    #
    #  We need:
    #    • A 300-day warm-up block ending at 2022-12-31
    #      (provides the first rolling window)
    #    • Forecast targets: all days from 2023-01-01 → end of dataset
    #
    #  Implementation:
    #    phase2_start_idx = index of first row in 2023
    #    We pass close[phase2_start_idx - WINDOW_SIZE : ] to the loop
    #    The loop's step=0 uses close[0:300] as its window
    #    and close[300] as its first forecast target → first 2023 day ✅

    TEST_START = pd.Timestamp("2023-01-01")

    phase2_start_idx = df[df["Date"] >= TEST_START].index[0]

    # Safety: ensure we have at least WINDOW_SIZE days before test start
    if phase2_start_idx < WINDOW_SIZE:
        raise ValueError(
            f"Not enough warm-up data before {TEST_START.date()}. "
            f"Need at least {WINDOW_SIZE} prior rows, found {phase2_start_idx}."
        )

    # Slice: warm-up (300 days of 2022) + all of 2023-2025
    phase2_df    = df.iloc[phase2_start_idx - WINDOW_SIZE :].reset_index(drop=True)
    phase2_close = phase2_df["Close"].values.astype(np.float64)
    phase2_dates = phase2_df["Date"].values

    warmup_end   = phase2_df["Date"].iloc[WINDOW_SIZE - 1].date()
    test_first   = phase2_df["Date"].iloc[WINDOW_SIZE].date()
    test_last    = phase2_df["Date"].iloc[-1].date()
    n_test_steps = len(phase2_close) - WINDOW_SIZE - 1

    print(f"\n[SPLIT] Phase 1 training period : 2015-01-01 → 2022-12-31")
    print(f"[SPLIT] Warm-up window ends     : {warmup_end}  (300 days, not forecasted)")
    print(f"[SPLIT] Phase 2 forecast starts : {test_first}  ← first OOS prediction")
    print(f"[SPLIT] Phase 2 forecast ends   : {test_last}")
    print(f"[SPLIT] Total OOS steps/run     : {n_test_steps}")
    print(f"[SPLIT] Total OOS steps × 5 runs: {n_test_steps * N_RERUNS}")

    assert n_test_steps > 50, \
        f"Too few OOS steps ({n_test_steps}). Check dataset dates."

    # ── 3. Load Phase 1 pretrained models (once, deepcopied per run) ─
    models_init = load_phase1_models(MODEL_DIR)

    # ── 4. Run 5 independent adaptive forecasts (OOS only) ──────────
    print(f"\n[PHASE 2] Starting {N_RERUNS} robustness runs…")
    run_results = []
    for run_id in range(1, N_RERUNS + 1):
        result = run_single_adaptive_forecast(
            close      = phase2_close,   # warm-up + OOS slice
            dates      = phase2_dates,
            models_init= models_init,
            run_id     = run_id,
        )
        run_results.append(result)

    # ── 5. Aggregate metrics: Mean ± Std over 5 runs ─────────────────
    adaptive_agg = aggregate_runs(run_results, col="Predicted")
    static_agg   = aggregate_runs(run_results, col="Static_Pred")

    # ── 6. Print consolidated results table ──────────────────────────
    print_results_table(adaptive_agg, static_agg)
    print_run_summary(run_results)

    # ── 7. Regime-wise breakdown ─────────────────────────────────────
    regime_df = build_regime_table(run_results)
    print("\n[REGIME ANALYSIS]  (mean across 5 runs)")
    print(regime_df.to_string(index=False))

    # ── 8. Save all output files ──────────────────────────────────────
    print("\n[SAVE] Writing output files…")
    save_all_outputs(run_results, adaptive_agg, static_agg, regime_df, SAVE_DIR)

    # ── 9. Research-grade plots ───────────────────────────────────────
    print("\n[PLOT] Generating visualisations…")
    plot_phase2_results(run_results, adaptive_agg, static_agg, SAVE_DIR)

    # ── 10. Final console summary ─────────────────────────────────────
    print("\n" + "═" * 65)
    print("  PHASE 2 COMPLETE ✅")
    print("═" * 65)
    print(f"  Evaluation period        : {test_first} → {test_last}")
    print(f"  OOS forecast steps/run   : {n_test_steps}")
    print(f"  Total runs               : {N_RERUNS}")
    print(f"  Avg Drifts / run         : "
          f"{np.mean([r['drift_count']   for r in run_results]):.1f}")
    print(f"  Avg Retrains / run       : "
          f"{np.mean([r['retrain_count'] for r in run_results]):.1f}")
    am, as_ = adaptive_agg["RMSE"]
    sm, ss  = static_agg["RMSE"]
    print(f"  Adaptive RMSE (mean±std) : {am:.5f} ± {as_:.5f}")
    print(f"  Static   RMSE (mean±std) : {sm:.5f} ± {ss:.5f}")
    da_a = adaptive_agg["DA"][0]
    da_s = static_agg["DA"][0]
    print(f"  Adaptive DA   (mean)     : {da_a:.2f}%")
    print(f"  Static   DA   (mean)     : {da_s:.2f}%")
    print("═" * 65)
    print("\n  Output files saved to /content/:")
    print("    phase2_forecast_log.csv        ← Date|Actual|Predicted|Error|…")
    print("    phase2_event_log.csv           ← Drift & retrain timestamps")
    print("    phase2_metrics_summary.csv     ← Mean±Std all metrics")
    print("    phase2_regime_analysis.csv     ← HIGH/LOW volatility breakdown")
    print("    phase2_robustness_5runs.csv    ← Per-run metric table")
    print("    phase2_adaptive_results.png    ← 6-panel research figure")
    print("═" * 65)

[DATA] Loading dataset…
[DATA] Total rows   : 2599
[DATA] Date range   : 2015-01-09 → 2025-07-25
[DATA] Close range  : 6970.55 → 26184.65

[SPLIT] Phase 1 training period : 2015-01-01 → 2022-12-31
[SPLIT] Warm-up window ends     : 2022-12-30  (300 days, not forecasted)
[SPLIT] Phase 2 forecast starts : 2023-01-02  ← first OOS prediction
[SPLIT] Phase 2 forecast ends   : 2025-07-25
[SPLIT] Total OOS steps/run     : 632
[SPLIT] Total OOS steps × 5 runs: 3160

[MODELS] Loading Phase 1 pretrained weights...
  [LOAD] IMF1       <- informer_IMF1.pt
       enc_seq=96 dec_seq=49 d_model=256 n_enc=2 n_dec=1
       STRICT load OK
  [LOAD] IMF2       <- informer_IMF2.pt
       enc_seq=96 dec_seq=49 d_model=256 n_enc=2 n_dec=1
       STRICT load OK
  [LOAD] IMF3       <- informer_IMF3.pt
       enc_seq=96 dec_seq=49 d_model=256 n_enc=2 n_dec=1
       STRICT load OK
  [LOAD] IMF4       <- lstm_IMF4.pt
       hidden=4 layers=4
       STRICT load OK
  [LOAD] IMF5       <- lstm_IMF5.pt
       hidden=4